In [1]:
workspace_id = "2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a"
bronze_lakehouse_id = "33b93b7e-0899-43c6-81a8-654bc740aa32"

StatementMeta(, 2bec0db1-aee2-4fbf-aec6-3ca9bcb3dff5, 3, Finished, Available, Finished, False)

In [2]:
import json
import time
import requests
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType, IntegerType

# --- CONFIGURATION & PARAMETERS ---
OPENAQ_API_KEY = "YOUR_API_KEY"
BRONZE_BASE_PATH = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/dbo"
NYC_BBOX = "-74.26,40.49,-73.70,40.92"  # New York City bounding box
DATE_FROM = "2024-01-01"
DATE_TO = "2026-01-01"
TARGET_PARAMETERS = {"pm25", "no2"}
MAX_SENSORS_TO_FETCH = 25

headers = {"X-API-Key": OPENAQ_API_KEY}

# 1. Fetch monitoring stations within NYC bounding box
locations_url = f"https://api.openaq.org/v3/locations?bbox={NYC_BBOX}&limit=100"
resp_loc = requests.get(locations_url, headers=headers, timeout=30)
resp_loc.raise_for_status()

locations = resp_loc.json().get("results") or []

if not locations:
    raise RuntimeError("OpenAQ API returned no monitoring stations for the NYC bounding box.")

json_rdd = spark.sparkContext.parallelize([json.dumps(r, ensure_ascii=False) for r in locations])
df_locations = spark.read.json(json_rdd)

df_locations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    f"{BRONZE_BASE_PATH}/openaq_locations_bronze"
)
print(f"Successfully saved {len(locations)} NYC stations to openaq_locations_bronze.")

# 2. Extract PM2.5 and NO2 sensors and fetch daily measurements
sensors_to_query = []
for loc in locations:
    loc_id = loc.get("id")
    loc_name = loc.get("name")
    for sensor in (loc.get("sensors") or []):
        param = sensor.get("parameter") or {}
        param_name = (param.get("name") or "").lower()
        if param_name in TARGET_PARAMETERS:
            sensors_to_query.append({
                "location_id": loc_id,
                "location_name": loc_name,
                "sensor_id": sensor.get("id"),
                "parameter_name": param_name,
                "parameter_units": param.get("units")
            })

sensors_to_query = sensors_to_query[:MAX_SENSORS_TO_FETCH]
print(f"Found {len(sensors_to_query)} PM2.5/NO2 sensors in NYC. Fetching daily measurements...")

measurement_rows = []
for s in sensors_to_query:
    sid = s["sensor_id"]
    days_url = (
        f"https://api.openaq.org/v3/sensors/{sid}/days"
        f"?date_from={DATE_FROM}&date_to={DATE_TO}&limit=1000"
    )
    r = requests.get(days_url, headers=headers, timeout=30)
    if r.status_code == 429:
        time.sleep(2)
        r = requests.get(days_url, headers=headers, timeout=30)
    if r.status_code != 200:
        print(f"Skipped sensor {sid} (HTTP {r.status_code})")
        continue

    daily_results = r.json().get("results") or []
    for item in daily_results:
        period_from = (item.get("period") or {}).get("datetimeFrom") or {}
        raw_dt = period_from.get("local") or period_from.get("utc") or ""
        date_str = raw_dt[:10]

        summary = item.get("summary") or {}
        coverage = item.get("coverage") or {}
        avg_val = summary.get("avg") if summary.get("avg") is not None else item.get("value")

        if not date_str or avg_val is None:
            continue

        measurement_rows.append((
            int(s["location_id"]),
            str(s["location_name"]),
            int(sid),
            str(s["parameter_name"]),
            str(s["parameter_units"] or ""),
            date_str,
            float(avg_val),
            float(summary["min"]) if summary.get("min") is not None else None,
            float(summary["max"]) if summary.get("max") is not None else None,
            int(coverage["observedCount"]) if coverage.get("observedCount") is not None else None
        ))
    time.sleep(0.25)

schema = StructType([
    StructField("location_id", LongType(), True),
    StructField("location_name", StringType(), True),
    StructField("sensor_id", LongType(), True),
    StructField("parameter_name", StringType(), True),
    StructField("parameter_units", StringType(), True),
    StructField("measurement_date", StringType(), True),
    StructField("avg_value", DoubleType(), True),
    StructField("min_value", DoubleType(), True),
    StructField("max_value", DoubleType(), True),
    StructField("observed_hours", IntegerType(), True)
])

df_measurements = spark.createDataFrame(measurement_rows, schema=schema)
df_measurements.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    f"{BRONZE_BASE_PATH}/openaq_measurements_bronze"
)

print(f"Successfully saved {df_measurements.count()} daily measurements to openaq_measurements_bronze.")
display(df_measurements.limit(10))

StatementMeta(, 2bec0db1-aee2-4fbf-aec6-3ca9bcb3dff5, 4, Finished, Available, Finished, False)

Successfully saved 76 NYC stations to openaq_locations_bronze.
Found 25 PM2.5/NO2 sensors in NYC. Fetching daily measurements...
Successfully saved 13581 daily measurements to openaq_measurements_bronze.


SynapseWidget(Synapse.DataFrame, d747603e-24ff-4edb-b85d-d292f30a9037)